In [22]:
%load_ext autoreload
%autoreload 2
import torch 
import torchvision
from torchvision import transforms
import pandas as pd 
import numpy as np 
from physics_utils import *
from dataset_dataloader import *
import yaml


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
temp_train_ds=DatasetMaker('../data/train_data.csv',transforms=transforms.Compose([FftTransform(),transforms.ToTensor()]))
all=[temp_train_ds[i][0] for i in range(len(temp_train_ds))]
a=torch.stack(all,dim=0)
train_mean=a[:,0,:,:].mean()
train_std=a[:,0,:,:].std()  ## add these stats to the config file for further use

In [ ]:
config_path = '../src/config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

config['stats']['train_mean'] = round(train_mean.item(), 4)
config['stats']['train_std'] = round(train_std.item(), 4)
config['fft_params']['notch_width']=5
config['fft_params']['notch_depth']=0.95
config['fft_params']['apply_bilateral']=False
config['data_path']['train']='../data/train_data.csv'
config['data_path']['test']='../data/test_data.csv'
config['data_path']['val']='../data/val_data.csv'

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print("YAML updated")

TypeError: 'NoneType' object is not subscriptable

In [ ]:
train_transform = transforms.Compose([FftTransform(width=NOTCH_WIDTH, notch_depth=NOTCH_DEPTH),
                                      transforms.RandomHorizontalFlip(p=0.5),
                                      transforms.RandomVerticalFlip(p=0.5),
                    # rotations (90 deg increase) to not destroy the fft transform
                                      transforms.RandomChoice([
                                      transforms.RandomRotation((0, 0)),
                                      transforms.RandomRotation((90, 90)),
                                      transforms.RandomRotation((180, 180)),
                                      transforms.RandomRotation((270, 270))]),
                                      transforms.ToTensor(),
                                      transforms.Normalize(mean=[TRAIN_MEAN], std=[TRAIN_STD])])

val_transform =transforms.Compose([FftTransform(width=NOTCH_WIDTH, notch_depth=NOTCH_DEPTH),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean=[TRAIN_MEAN], std=[TRAIN_STD])])

In [ ]:
train_ds=DatasetMaker(data_csv_path=TRAIN_DATA_PATH,transforms=train_transform)
val_ds=DatasetMaker(data_csv_path=VAL_DATA_PATH,transforms=val_transform)
test_ds=DatasetMaker(data_csv_path=TEST_DATA_PATH,transforms=val_transform)